# Milestone 1 — Notebook chung Khánh, Khang, Nguyên, Danh

**Chỉ một notebook này**, không tạo notebook thử riêng. Hàm nghiệp vụ viết trong .py.
Khánh: arXiv API/metadata; Khang: tải nguồn; Nguyên: tham khảo; Danh: tích hợp/metrics.
Hiện tại các hàm vẫn là prototype. Các ô mặc định đọc fixture, không gọi API.
Chạy không lỗi với fixture KHÔNG chứng minh crawler đã hoạt động.
Đọc docs/milestone1/PHAN_CONG_MILESTONE_1_4_NGUOI.md để xem quy trình mới.

Phát triển Jupyter local, benchmark Colab CPU, báo cáo nộp tiếng Anh.
Chưa có phạm vi bài thầy giao. Số liệu fixture không được dùng làm kết quả nộp.


## Quy trình dùng chung và kiểm tra

1. Mỗi người đọc schema __init__.py, sửa module .py của mình rồi chạy phần tương ứng bên dưới.
2. Fixture giúp làm trước khi đồng đội xong; sau triển khai bật cell gọi hàm thật của mình.
3. Sửa cờ RUN của mình thành True trong setup sau triển khai → Restart kernel → Run All; cờ phần khác False. Trước commit trả cờ về False.
4. Kiểm tra đủ 5 lớp: chạy được → đúng schema → đúng nội dung → xử lý lỗi → tích hợp được.
5. Danh ghép các hàm thật, chạy vài bài, sau đó chạy phạm vi chính thức và benchmark.

Chỉ sửa/lưu cell phần mình, phối hợp để tránh hai người lưu đè notebook. Làm module .py song song được,
nhưng ghép notebook lần lượt. Không vô tình Run All nhiều lần khi đã bật các call mạng.
Trước commit: clear output cào, không lưu API key. Nếu cần nạp lại module sau khi sửa, restart kernel.

**Checklist mỗi người ghi lại:** hàm/case đã thử, input, output mong đợi, output thực tế,
đường dẫn file tạo ra, lỗi/giới hạn còn lại. Prototype raise NotImplementedError là trạng thái chưa triển khai,
không được thay bằng trả fixture để giả thành công.


In [ ]:
from pathlib import Path
import sys
from copy import deepcopy
from functools import partial

# Chạy từ project root hoặc thư mục con; Colab clone repo trước.
SRC = next((p / 'src' for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'src' / 'crawling' / '__init__.py').is_file()), None)
assert SRC is not None, 'Mở notebook trong project hoặc đặt SRC thủ công.'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
from crawling import fixtures
from crawling.discover import discover_papers, fetch_paper, fetch_submission_dates
from crawling.download import download_version, prepare_source
from crawling.references import collect_references
from crawling.pipeline import run_pipeline
from crawling.metrics import summarize_corpus, summarize_benchmark

# Chỉ bật phần đã triển khai. Mặc định Run All không gọi crawler/mạng.
RUN_KHANH = False
RUN_KHANG = False
RUN_NGUYEN = False
USE_REAL_DATES = False
RUN_DANH_METRICS = False
RUN_INTEGRATION = False
POLICY = deepcopy(fixtures.HTTP_POLICY)
TEST_ROOT = SRC.parent / 'data' / 'manual'


## 1. Khánh — Input phạm vi, output danh sách bài và metadata
`Scope` -> `DiscoveryResult`; lấy danh sách tại result['papers'].
Metadata trong mỗi PaperRecord được Danh lưu ra file. Fixture chỉ một bài để dễ kiểm tra.


**Quy trình Khánh:** fetch_paper → discover_papers → fetch_submission_dates. Dùng arXiv API requests + Atom/XML. Kiểm tra lịch sử phiên bản, metadata, mã thiếu/API lỗi và output DateLookupResult. Sau khi chạy thật, đối chiếu nội dung với arXiv, không chỉ kiểm tra không có exception.


In [ ]:
khanh_scope_sample = deepcopy(fixtures.SCOPE)
khanh_discovery_sample = deepcopy(fixtures.DISCOVERY_RESULT)
assert khanh_discovery_sample['scope'] == khanh_scope_sample
assert khanh_discovery_sample['papers'][0]['versions'] == [1, 2]  # fixture, không ràng buộc bài thật
assert khanh_discovery_sample['papers'][0]['metadata']['arxiv_id'] == '1810.04805'
khanh_discovery_sample


In [ ]:
if RUN_KHANH:
    khanh_test_scope = {'mode': 'ids', 'ids': ['1810.04805', '1907.11692'],
                        'start_id': None, 'end_id': None}
    khanh_live = discover_papers(khanh_test_scope, policy=POLICY)
    assert khanh_live['requested_count'] == 2
    assert khanh_live['status'] in {'success', 'partial', 'failed'}
    khanh_accounted = [p['arxiv_id'] for p in khanh_live['papers']] + [e['arxiv_id'] for e in khanh_live['issues']]
    assert sorted(khanh_accounted) == sorted(khanh_test_scope['ids'])
    for khanh_record in khanh_live['papers']:
        assert khanh_record['arxiv_id'] == khanh_record['metadata']['arxiv_id']
        assert khanh_record['versions'] == sorted(set(khanh_record['versions']))
        assert khanh_record['versions'][0] == 1
    khanh_dates_live = fetch_submission_dates(['1706.03762'], policy=POLICY)
    assert set(khanh_dates_live['dates']) == {'1706.03762'}
    assert fetch_submission_dates([], policy=POLICY) == {'dates': {}, 'errors': {}}
    print(khanh_live, khanh_dates_live)
else:
    print('Khánh: đang tắt gọi hàm thật. Đọc mục 12 phân công trước khi bật.')


### Khánh: kiểm tra tay và ghi bằng chứng

Đọc mục 12 của `docs/milestone1/PHAN_CONG_MILESTONE_1_4_NGUOI.md`.
Restart Kernel → chạy setup → các ô phần này là đủ, không cần fixture của phần khác.
Các assert trên chỉ kiểm một số bất biến; còn phải thêm ca lỗi/schema và mở đối chiếu nguồn/file.

| Ngày/môi trường | Hàm/case | Input/nguồn đối chiếu | Mong đợi | Thực tế/file | Kết luận/giới hạn |
|---|---|---|---|---|---|
| Chưa chạy thật | Điền sau triển khai | — | — | — | Chưa kiểm chứng |

Giữ mặc định cờ RUN=False trước commit, clear output. Không ghi pass cho ca chưa chạy.


## 2. Khang — Tải và xử lý nguồn độc lập discovery

Chạy setup chung → fixture Khang → hai cell kiểm thử offline → cell tải thật nếu muốn.
Các test gọi **hàm thật** với gói local hoặc HTTP giả; không dùng số liệu fixture làm kết quả.
`RUN_KHANG=False` mặc định chỉ tắt cell mạng; hai cell test offline vẫn chạy.
Mỗi lần kiểm thử tạo một thư mục `data/manual/khang/local-*` mới, không ghi lên corpus.
Cell mạng lấy ID/version từ fixture, tải v1/v2 vào các thư mục riêng và kiểm cache.
Không cần chạy phần Khánh/Nguyên/Danh. Đọc hướng dẫn và bằng chứng ở cuối phần này.


In [ ]:
# Độc lập phần Khánh: lấy trực tiếp fixture, không lấy biến của section khác.
khang_paper_sample = deepcopy(fixtures.PAPER_BERT)
khang_id = khang_paper_sample['arxiv_id']
khang_version = khang_paper_sample['versions'][0]
khang_version_dir = TEST_ROOT / 'khang' / '1810-04805' / 'sources' / f'v{khang_version}'
khang_download_sample = deepcopy(fixtures.DOWNLOAD_RESULT)
assert (khang_download_sample['arxiv_id'], khang_download_sample['version']) == (khang_id, khang_version)
assert khang_download_sample['source_bytes_before'] >= khang_download_sample['source_bytes_after']
{'input': (khang_id, khang_version, str(khang_version_dir)), 'sample': khang_download_sample}


In [ ]:
# Kiểm thử OFFLINE: không gọi HTTP, giữ các file nhỏ để mở kiểm tra tay.
import gzip
import hashlib
import io
import json
import tarfile
import tempfile
import unittest
import zipfile
from unittest.mock import patch
import crawling.download as khang_module

(TEST_ROOT / 'khang').mkdir(parents=True, exist_ok=True)
khang_local_root = Path(tempfile.mkdtemp(prefix='local-', dir=TEST_ROOT / 'khang'))
khang_tex = ('\\documentclass{article}\r\n\\begin{document}\r\n'
             'Tiếng Việt: giữ nguyên nội dung.\r\n\\input{sections/intro}\r\n'
             '\\begin{figure}\\includegraphics{figures/chart.png}\\end{figure}\r\n'
             '\\begin{tabular}{c}A\\\\B\\end{tabular}\r\n\\end{document}\r\n').encode('utf-8')
khang_texts = {
    'main.tex': khang_tex, 'sections/intro.tex': b'\\section{Intro}\nOriginal text.\n',
    'refs.bib': b'@article{example,title={Example}}\n', 'style.sty': b'\\ProvidesPackage{style}\n',
    'class.cls': b'\\ProvidesClass{class}\n', 'main.bbl': b'\\bibitem{example} Example\n',
    'README': b'Keep this extensionless text unchanged.\n',
    'legacy.txt': b'caf\xe9\r\n', 'vector.svg': b'<svg xmlns="http://www.w3.org/2000/svg"/>',
}
khang_binary = {'figures/chart.png': b'\x89PNG\r\n\x1a\n\x00binary image',
                'disguised.tex': b'\xff\xd8\xff\x00binary JPEG'}

def khang_tar(items, mode='w:gz'):
    stream = io.BytesIO()
    with tarfile.open(fileobj=stream, mode=mode) as archive:
        for name, value in items:
            member = tarfile.TarInfo(name)
            if isinstance(value, bytes):
                member.size = len(value)
                archive.addfile(member, io.BytesIO(value))
            else:
                member.type, member.linkname = value
                archive.addfile(member)
    return stream.getvalue()

def khang_pdf():
    # Small valid PDF, with computed xref offsets; no PDF dependency required.
    result = b'%PDF-1.4\n'
    offsets = [0]
    for number, body in enumerate([
        b'<< /Type /Catalog /Pages 2 0 R >>',
        b'<< /Type /Pages /Kids [3 0 R] /Count 1 >>',
        b'<< /Type /Page /Parent 2 0 R /MediaBox [0 0 100 100] >>',
    ], 1):
        offsets.append(len(result))
        result += str(number).encode() + b' 0 obj\n' + body + b'\nendobj\n'
    xref = len(result)
    result += b'xref\n0 4\n0000000000 65535 f \n'
    result += b''.join(f'{offset:010d} 00000 n \n'.encode() for offset in offsets[1:])
    return result + f'trailer\n<< /Size 4 /Root 1 0 R >>\nstartxref\n{xref}\n%%EOF\n'.encode()

class KhangLocalTests(unittest.TestCase):
    def setUp(self):
        self.root = khang_local_root / self._testMethodName
        self.root.mkdir()
        self.output = self.root / 'sources' / 'v1'

    def prepare(self, payload):
        source = self.root / 'source.payload'  # intentionally no format extension
        source.write_bytes(payload)
        result = prepare_source(source, self.output, arxiv_id=khang_id, version=khang_version)
        self.assertEqual(set(result), set(fixtures.DOWNLOAD_RESULT))
        self.assertEqual(result['attempts'], 0)
        self.assertIs(result['used_cache'], False)
        self.assertEqual(result['archive_bytes'], len(payload))
        return result

    def test_single_tex_and_gzip(self):
        for payload in (khang_tex, gzip.compress(khang_tex)):
            result = self.prepare(payload)
            self.assertEqual(result['status'], 'success')
            self.assertEqual((self.output / 'main.tex').read_bytes(), khang_tex)
            self.assertEqual(result['source_bytes_before'], len(khang_tex))
            self.assertEqual(result['source_bytes_after'], len(khang_tex))
            self.assertEqual(result['file_count'], 1)

    def test_tar_hashes_sizes_and_structure(self):
        for mode in ('w', 'w:gz'):
            result = self.prepare(khang_tar(list((khang_texts | khang_binary).items()), mode))
            self.assertEqual(result['status'], 'success')
            self.assertEqual(result['source_bytes_before'], sum(map(len, (khang_texts | khang_binary).values())))
            self.assertEqual(result['source_bytes_after'], sum(map(len, khang_texts.values())))
            self.assertEqual(result['file_count'], len(khang_texts))
            actual = {p.relative_to(self.output).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest()
                      for p in self.output.rglob('*') if p.is_file()}
            self.assertEqual(actual, {name: hashlib.sha256(content).hexdigest() for name, content in khang_texts.items()})
            self.assertIn(b'\\includegraphics', (self.output / 'main.tex').read_bytes())
            self.assertFalse((self.output.parent / '.v1.receipt.json').exists())

    def test_pdf_and_no_tex_archive(self):
        for payload, before in [(khang_pdf(), len(khang_pdf())),
                                (gzip.compress(khang_pdf()), len(khang_pdf())),
                                (khang_tar([('paper.pdf', khang_pdf())]), len(khang_pdf()))]:
            self.prepare(khang_tex)  # no_tex replaces an existing source with an empty record
            result = self.prepare(payload)
            self.assertEqual(result['status'], 'no_tex')
            self.assertEqual(result['source_bytes_before'], before)
            self.assertEqual((result['source_bytes_after'], result['file_count']), (0, 0))
            self.assertEqual(list(self.output.iterdir()), [])

    def test_word_only(self):
        stream = io.BytesIO()
        members = {'[Content_Types].xml': b'<Types/>',
                   'word/document.xml': b'<w:document xmlns:w="http://schemas.openxmlformats.org/wordprocessingml/2006/main"><w:body/></w:document>'}
        with zipfile.ZipFile(stream, 'w', zipfile.ZIP_DEFLATED) as archive:
            for name, content in members.items():
                archive.writestr(name, content)
        result = self.prepare(stream.getvalue())
        self.assertEqual(result['status'], 'no_tex')
        self.assertEqual(result['source_bytes_before'], sum(map(len, members.values())))

    def test_damage_and_unknown_preserve_good_output(self):
        self.prepare(khang_tex)
        payloads = [b'random text', b'\x00\x01random', b'%PDF-1.4\ntruncated',
                    b'<html>Service unavailable</html>', gzip.compress(khang_tex)[:-5],
                    khang_tar([('main.tex', khang_tex)], 'w')[:600], b'PK\x03\x04bad', b'']
        for payload in payloads:
            with self.subTest(payload=payload[:12]):
                result = self.prepare(payload)
                self.assertEqual(result['status'], 'failed')
                self.assertIsNone(result['source_bytes_after'])
                self.assertEqual((self.output / 'main.tex').read_bytes(), khang_tex)

    def test_paths_links_and_duplicates(self):
        self.prepare(khang_tex)
        bad = [[(name, khang_tex)] for name in ('../escape.tex', '/escape.tex', 'C:/escape.tex',
               'folder\\escape.tex', 'file:stream', 'NUL.tex', 'a/../../escape.tex')]
        bad += [[('link', (kind, '../escape.tex'))] for kind in (tarfile.SYMTYPE, tarfile.LNKTYPE)]
        bad += [[('pipe', (tarfile.FIFOTYPE, ''))], [('main.tex', khang_tex), ('main.tex', khang_tex)],
                [('A/main.tex', khang_tex), ('a/other.tex', khang_tex)],
                [('a', khang_tex), ('a/main.tex', khang_tex)]]
        for members in bad:
            with self.subTest(names=[name for name, _ in members]):
                self.assertEqual(self.prepare(khang_tar(members))['status'], 'failed')
                self.assertFalse((self.output.parent / 'escape.tex').exists())
                self.assertEqual((self.output / 'main.tex').read_bytes(), khang_tex)

    def test_limits_stop_before_commit(self):
        self.prepare(khang_tex)
        for constant, limit, payload in [
            ('_MAX_ARCHIVE_BYTES', 10, khang_tex),
            ('_MAX_EXPANDED_BYTES', 1024, gzip.compress(b'A' * 100_000)),
            ('_MAX_FILE_BYTES', 32, khang_tar([('main.tex', khang_tex)])),
            ('_MAX_MEMBERS', 1, khang_tar([('a.tex', khang_tex), ('b.tex', khang_tex)])),
        ]:
            with self.subTest(limit=constant), patch.object(khang_module, constant, limit):
                self.assertEqual(self.prepare(payload)['status'], 'failed')
                self.assertEqual((self.output / 'main.tex').read_bytes(), khang_tex)

    def test_invalid_inputs(self):
        with self.assertRaises(ValueError):
            prepare_source(self.root / 'missing', self.output, arxiv_id=khang_id, version=1)
        source = self.root / 'source'
        source.write_bytes(khang_tex)
        for ident, version in [('1810.04805v1', 1), ('../bad', 1), (khang_id, 0), (khang_id, True)]:
            with self.assertRaises(ValueError):
                prepare_source(source, self.output, arxiv_id=ident, version=version)
        with self.assertRaises(ValueError):
            prepare_source(source, source, arxiv_id=khang_id, version=1)

khang_local_report = unittest.TextTestRunner(verbosity=2).run(unittest.defaultTestLoader.loadTestsFromTestCase(KhangLocalTests))
assert khang_local_report.wasSuccessful(), 'Có lỗi kiểm thử local; đọc traceback phía trên.'
print('Mở kiểm tra tay:', khang_local_root / 'test_tar_hashes_sizes_and_structure' / 'sources' / 'v1')


In [ ]:
# Fake HTTP chỉ trong cell kiểm thử. requests.get bị thay hoàn toàn: KHÔNG gọi mạng.
import requests
from contextlib import contextmanager
from types import SimpleNamespace
from unittest.mock import Mock

@contextmanager
def khang_fake_sleep():
    # Only replace the downloader clock; Jupyter background threads keep real sleep.
    clock = SimpleNamespace(monotonic=khang_module.time.monotonic, sleep=Mock())
    with patch.object(khang_module, "time", clock):
        yield clock.sleep

from email.utils import format_datetime
from datetime import datetime, timedelta, timezone

class KhangResponse:
    def __init__(self, content=khang_tex, status=200, headers=None):
        self.status_code = status
        self.headers = headers or {}
        self.raw = io.BytesIO(content)
    def __enter__(self):
        return self
    def __exit__(self, *args):
        self.raw.close()

class KhangHTTPTests(unittest.TestCase):
    def setUp(self):
        self.root = khang_local_root / self._testMethodName
        self.root.mkdir()
        self.output = self.root / 'sources' / 'v1'
        self.policy = deepcopy(fixtures.HTTP_POLICY)

    def download(self, **kwargs):
        return download_version(khang_id, khang_version, self.output, policy=self.policy, **kwargs)

    def snapshot(self):
        return {p.relative_to(self.output.parent).as_posix(): p.read_bytes()
                for p in self.output.parent.rglob('*') if p.is_file()}

    def test_exact_version_and_cache(self):
        with patch.object(khang_module.requests, 'get', return_value=KhangResponse()) as get, khang_fake_sleep() as sleep:
            result = self.download()
            self.assertEqual(result['status'], 'success')
            get.assert_called_once()
            self.assertEqual(get.call_args.args[0], f'https://export.arxiv.org/src/{khang_id}v{khang_version}')
            self.assertEqual(get.call_args.kwargs['timeout'], self.policy['timeout_seconds'])
            self.assertFalse(get.call_args.kwargs['allow_redirects'])
            sleep.assert_called_once_with(3.0)
            before = self.snapshot()
            cached = self.download()
            self.assertTrue(cached['used_cache'])
            self.assertEqual(cached['attempts'], 0)
            self.assertEqual(self.snapshot(), before)
            self.assertEqual(get.call_count, 1)
            self.assertEqual(cached['source_bytes_before'], len(khang_tex))

    def test_bad_cache_refresh(self):
        with khang_fake_sleep():
            for mutation in ('missing-receipt', 'json', 'identity', 'version', 'kind', 'schema', 'stats', 'hash', 'modified', 'missing', 'extra'):
                with self.subTest(mutation=mutation):
                    with patch.object(khang_module.requests, 'get', return_value=KhangResponse()):
                        self.assertEqual(self.download(use_cache=False)['status'], 'success')
                    receipt_path = self.output.parent / '.v1.receipt.json'
                    receipt = json.loads(receipt_path.read_text())
                    if mutation == 'missing-receipt':
                        receipt_path.unlink()
                    elif mutation == 'json':
                        receipt_path.write_text('{')
                    elif mutation in ('identity', 'version', 'kind', 'schema', 'stats', 'hash'):
                        if mutation == 'identity': receipt['arxiv_id'] = '1907.11692'
                        if mutation == 'version': receipt['version'] = True
                        if mutation == 'kind': receipt['source_kind'] = 'unknown'
                        if mutation == 'schema': receipt['schema_version'] = '0'
                        if mutation == 'stats': receipt['result']['source_bytes_after'] = 0
                        if mutation == 'hash': receipt['files_sha256']['main.tex'] = '0' * 64
                        receipt_path.write_text(json.dumps(receipt))
                    elif mutation == 'modified': (self.output / 'main.tex').write_bytes(b'bad')
                    elif mutation == 'missing': (self.output / 'main.tex').unlink()
                    elif mutation == 'extra': (self.output / 'extra').write_bytes(b'extra')
                    with patch.object(khang_module.requests, 'get', return_value=KhangResponse()) as get:
                        result = self.download()
                        self.assertEqual(result['status'], 'success')
                        self.assertFalse(result['used_cache'])
                        get.assert_called_once()

    def test_retries_and_retry_after(self):
        scenarios = [([KhangResponse(status=429, headers={'Retry-After': '70'}), KhangResponse()], [3.0, 70.0]),
                     ([requests.Timeout('secret must not appear'), KhangResponse()], [3.0, 3.0]),
                     ([KhangResponse(status=503), KhangResponse(status=503), KhangResponse()], [3.0, 3.0, 3.0])]
        for responses, waits in scenarios:
            with patch.object(khang_module.requests, 'get', side_effect=responses) as get, khang_fake_sleep() as sleep:
                result = self.download(use_cache=False)
                self.assertEqual(result['status'], 'success')
                self.assertEqual(result['attempts'], len(waits))
                self.assertEqual([call.args[0] for call in sleep.call_args_list], waits)
        header = format_datetime(datetime.now(timezone.utc) + timedelta(seconds=120), usegmt=True)
        with patch.object(khang_module.requests, 'get', side_effect=[KhangResponse(status=429, headers={'Retry-After': header}), KhangResponse()]), khang_fake_sleep() as sleep:
            self.assertEqual(self.download(use_cache=False)['status'], 'success')
            self.assertGreater(sleep.call_args_list[1].args[0], 100)

    def test_failure_preserves_sources_and_receipt(self):
        with khang_fake_sleep():
            with patch.object(khang_module.requests, 'get', return_value=KhangResponse()):
                self.download()
            before = self.snapshot()
            with patch.object(khang_module.requests, 'get', side_effect=requests.Timeout('API_KEY=private')) as get:
                result = self.download(use_cache=False)
                self.assertEqual(result['status'], 'failed')
                self.assertEqual(result['attempts'], self.policy['max_attempts'])
                self.assertNotIn('private', result['error'])
                self.assertEqual(self.snapshot(), before)
            for response in (KhangResponse(status=404), KhangResponse(status=302),
                             KhangResponse(b'<html>Error</html>'), KhangResponse(b'bad gzip\x00'),
                             KhangResponse(headers={'Content-Length': '999999999999'})):
                with patch.object(khang_module.requests, 'get', return_value=response) as get:
                    self.assertEqual(self.download(use_cache=False)['status'], 'failed')
                    self.assertEqual(self.snapshot(), before)
                    get.assert_called_once()

    def test_pdf_receipt_and_local_invalidation(self):
        with patch.object(khang_module.requests, 'get', return_value=KhangResponse(khang_pdf())) as get, khang_fake_sleep():
            result = self.download()
            self.assertEqual(result['status'], 'no_tex')
            self.assertTrue(self.download()['used_cache'])
            get.assert_called_once()
        source = self.root / 'local.tex'
        source.write_bytes(khang_tex)
        self.assertEqual(prepare_source(source, self.output, arxiv_id=khang_id, version=1)['status'], 'success')
        self.assertFalse((self.output.parent / '.v1.receipt.json').exists())

    def test_commit_failure_rolls_back(self):
        with khang_fake_sleep():
            with patch.object(khang_module.requests, 'get', return_value=KhangResponse()):
                self.download()
            before = self.snapshot()
            real_replace = khang_module.os.replace
            def fail_receipt(source, destination):
                if Path(source).name == 'receipt.json':
                    raise OSError('simulated disk error')
                return real_replace(source, destination)
            with patch.object(khang_module.requests, 'get', return_value=KhangResponse(khang_tex + b'changed')), patch.object(khang_module.os, 'replace', side_effect=fail_receipt):
                self.assertEqual(self.download(use_cache=False)['status'], 'failed')
            self.assertEqual(self.snapshot(), before)

    def test_legacy_id_and_policy_validation(self):
        with patch.object(khang_module.requests, 'get', return_value=KhangResponse()) as get, khang_fake_sleep():
            result = download_version('hep-th/9901001', 2, self.output.parent / 'v2', policy=self.policy)
            self.assertEqual(result['arxiv_id'], 'hep-th/9901001')
            self.assertTrue(get.call_args.args[0].endswith('/hep-th/9901001v2'))
        with patch.object(khang_module.requests, 'get') as get:
            for key, value in [('timeout_seconds', 0), ('max_attempts', True), ('arxiv_delay_seconds', float('nan')), ('user_agent', 'bad\nheader')]:
                with self.subTest(key=key), self.assertRaises(ValueError):
                    download_version(khang_id, 1, self.output, policy=self.policy | {key: value})
            get.assert_not_called()

khang_http_report = unittest.TextTestRunner(verbosity=2).run(unittest.defaultTestLoader.loadTestsFromTestCase(KhangHTTPTests))
assert khang_http_report.wasSuccessful(), 'Có lỗi kiểm thử HTTP/cache; đọc traceback phía trên.'


In [ ]:
# Chỉ cell này gọi mạng; RUN_KHANG=False trong setup là mặc định.
# v1/v2 lấy từ fixture để phát triển độc lập, không phải discovery thật.
if RUN_KHANG:
    khang_live_results = []
    for khang_v in khang_paper_sample['versions']:
        khang_dest = TEST_ROOT / 'khang' / 'live' / khang_id.replace('.', '-').replace('/', '-') / 'sources' / f'v{khang_v}'
        khang_live = download_version(khang_id, khang_v, khang_dest, policy=POLICY, use_cache=True)
        assert (khang_live['arxiv_id'], khang_live['version']) == (khang_id, khang_v)
        assert khang_live['status'] in {'success', 'no_tex', 'failed'}
        if khang_live['status'] != 'failed':
            assert khang_live['source_bytes_after'] == sum(p.stat().st_size for p in khang_dest.rglob('*') if p.is_file())
            khang_cached = download_version(khang_id, khang_v, khang_dest, policy=POLICY, use_cache=True)
            assert khang_cached['used_cache'] and khang_cached['attempts'] == 0
        khang_live_results.append(khang_live)
        print(khang_dest, khang_live)
else:
    print('Khang: mạng đang tắt. Các kiểm thử local/fake HTTP phía trên vẫn chạy được.')


### Khang: hướng dẫn kiểm tra tay từ lúc mở Ubuntu

**Dành cho máy hiện tại:** Windows + Ubuntu 24.04 WSL; Miniconda ở
`/home/khangtran/miniconda3`; môi trường `arxiv-project` đã có.
Conda dùng bằng lệnh trong terminal Ubuntu, không cần mở một ứng dụng Conda riêng
và không cần cài lại package mỗi lần làm việc.

**Bước 1 — Mở Ubuntu.**

Trong Windows, bấm Start, tìm **Ubuntu 24.04** rồi mở. Hoặc mở Windows Terminal,
chọn tab Ubuntu; nếu đang ở PowerShell thì chạy `wsl -d Ubuntu-24.04` để vào Ubuntu.
Các lệnh Bash bên dưới phải chạy trong Ubuntu. Dấu nhắc thường có dạng
`khangtran@...:~$`; không chép phần dấu nhắc `$` vào lệnh.

**Bước 2 — Kích hoạt Conda và vào folder project.**

Chép nguyên khối dưới vào terminal Ubuntu, nhấn Enter:

```bash
source ~/miniconda3/etc/profile.d/conda.sh
conda activate arxiv-project
cd /mnt/c/Users/KhangTran/Downloads/HCMUS/Nhap_Mon_KHDL/arXivProject
```

`source ...` nạp lệnh Conda cho terminal hiện tại, kể cả khi trước đó báo
`conda: command not found`. Khi activate thành công, đầu dòng thường hiện
`(arxiv-project)`. `/mnt/c/...` chính là folder ở ổ C: của Windows, không phải
một bản clone khác. Kiểm tra:

```bash
pwd
python -c "import sys; print(sys.executable)"
```

`pwd` phải kết thúc bằng `arXivProject`; Python phải nằm trong
`/home/khangtran/miniconda3/envs/arxiv-project/bin/python`.
Nếu báo môi trường không tồn tại, chạy `conda env list` để xem lại. Chỉ khi thật sự
chưa có môi trường, từ folder project mới chạy `conda env create -f environment.yml`,
rồi `conda activate arxiv-project`.

**Bước 3 — Mở JupyterLab và notebook.**

Trong chính terminal đã activate và cd ở bước 2, chạy:

```bash
python -m jupyterlab
```

Giữ terminal này mở vì nó đang chạy Jupyter. Nếu trình duyệt Windows không tự mở,
chép đường dẫn `http://localhost:8888/...token=...` hoặc `http://127.0.0.1:8888/...`
do terminal in ra vào Chrome/Edge. Dùng **đúng cổng và toàn bộ token vừa in**, không
tự gõ token mẫu; cổng có thể là 8889 nếu 8888 đang bận.

Trong trang JupyterLab, dùng cây file bên trái mở lần lượt
`src` → `notebooks` → nhấp đúp **milestone1.ipynb**. Khởi chạy từ root project như trên
để cây file có cả `src` lẫn `data`. Nếu cây file chỉ hiện notebook, dùng Explorer theo
bước 9 để mở dữ liệu; lần sau khởi chạy lại từ root project. Chọn kernel **Python 3 (ipykernel)**
nếu được hỏi. Với Jupyter khởi chạy từ môi trường ở bước 2, đây là kernel Python của
môi trường đó trong cấu hình đã kiểm thử. Nếu máy có nhiều kernel và không chắc,
mở **File → New → Console**, chọn cùng kernel rồi chạy `import sys; print(sys.executable)`;
đường dẫn cần thuộc `envs/arxiv-project`. Console chỉ để kiểm tra, không tạo notebook thứ hai.

**Bước 4 — Hiểu cách chạy một cell.**

Một cell code có ô `In [ ]` ở bên trái. Nhấp vào cell, bấm **Shift+Enter** để chạy
và chuyển xuống cell tiếp theo. `[*]` nghĩa là đang chạy; khi thành `[1]`, `[2]`…
thì cell đã xong. Kết quả nằm ngay phía dưới cell. Cell Markdown là chữ hướng dẫn;
Shift+Enter chỉ hiển thị chữ đó, không chạy Python.

Nếu có traceback đỏ hoặc cuối test ghi `FAILED`/`ERROR`, dừng ở cell đó và đọc lỗi.
Dòng `status='failed'` của một **ca lỗi được cố ý giả lập** không đồng nghĩa test thất bại:
test đạt khi cuối nhóm vẫn ghi `OK`.

**Bước 5 — Chạy setup chung, giữ mạng tắt.**

Chọn **Kernel → Restart Kernel…**, xác nhận restart. Sau đó tìm cell code gần đầu
notebook bắt đầu bằng `from pathlib import Path`; đây là cell import/cấu hình chung.
Đảm bảo các cờ `RUN_KHANH`, `RUN_KHANG`, `RUN_NGUYEN`, `RUN_DANH_METRICS`,
`RUN_INTEGRATION` đều là `False`; giữ `USE_REAL_DATES=False`. Chạy cell bằng Shift+Enter.
Không có output là bình thường nếu cell không báo lỗi.

**Bước 6 — Chạy phần Khang theo đúng thứ tự, chưa gọi mạng.**

Cuộn đến **“2. Khang — Tải và xử lý nguồn độc lập discovery”**. Không cần chạy
cell của Khánh, Nguyên hay Danh. Chạy lần lượt bốn cell code của Khang:

| Thứ tự | Nhận ra cell bằng dòng đầu | Thao tác và kết quả cần thấy |
|---|---|---|
| 1 | `# Độc lập phần Khánh: lấy trực tiếp fixture...` | Shift+Enter. Cell đặt ID/version và hiện input/output mẫu. Các byte trong mẫu là giả, chưa phải tải thật. |
| 2 | `# Kiểm thử OFFLINE: không gọi HTTP...` | Shift+Enter, đợi xong. Cuối nhóm có `Ran 8 tests ... OK` và dòng `Mở kiểm tra tay: ...`. Đây là kiểm thử hàm `prepare_source` thật bằng gói nhỏ tự tạo. |
| 3 | `# Fake HTTP chỉ trong cell kiểm thử...` | Shift+Enter, đợi `Ran 7 tests ... OK`. Cell giả lập HTTP để kiểm retry/cache/lỗi ghi file; không gửi request ra mạng. |
| 4 | `# Chỉ cell này gọi mạng; RUN_KHANG=False...` | Shift+Enter. Với cờ đang False, hiện `Khang: mạng đang tắt...`; đây là hành vi đúng. |

Mỗi lần chạy cell offline sẽ tạo một folder `local-*` mới, nên xem đường dẫn vừa in
của **lần hiện tại**. Để chạy lại cả nhóm test, chạy lại cell 2 trước cell 3; không chạy
riêng cell 3 hai lần với cùng thư mục test đã có. Khi muốn kiểm toàn notebook từ đầu,
giữ các cờ False rồi dùng **Kernel → Restart Kernel and Run All Cells…**; vẫn không gọi mạng.

**Bước 7 — Mở file của bài kiểm thử nhỏ và kiểm bằng mắt.**

Trong cây file bên trái JupyterLab, quay về root project rồi mở
`data` → `manual` → `khang` → folder `local-*` vừa được in →
`test_tar_hashes_sizes_and_structure` → `sources` → `v1`.
Nếu không thấy folder mới, bấm nút refresh cây file. Mở file bằng nhấp phải →
**Open With → Editor** nếu nhấp đúp không mở dạng text.

Kiểm từng điểm:

1. Mở `main.tex`: chữ Việt còn đúng; `\input{sections/intro}`, `\includegraphics{figures/chart.png}`,
   môi trường `figure` và `tabular` vẫn còn. Không yêu cầu compile PDF vì ảnh đã bị loại có chủ ý.
2. Mở `sections/intro.tex`: folder con và nội dung còn nguyên.
3. Kiểm có `refs.bib`, `style.sty`, `class.cls`, `main.bbl`, `README`, `legacy.txt`, `vector.svg`.
4. Không còn `figures/chart.png` hay `disguised.tex` (file giả đuôi tex nhưng nội dung JPEG).
   Folder `figures` có thể không tồn tại vì không còn file được giữ trong đó.
5. Cell offline đã so SHA-256 **từng file** và tự cộng byte; `OK` nghĩa các phép so đó đạt.
   Không kết luận byte nguyên vẹn chỉ từ việc mở nhìn vài dòng.
6. Ở folder `local-*`, mở `test_pdf_and_no_tex_archive/sources/v1`: thư mục rỗng là đúng
   cho nguồn chỉ PDF. Trong `test_damage_and_unknown_preserve_good_output/sources/v1`,
   `main.tex` tốt vẫn còn sau khi thử các payload hỏng.

**Bước 8 — Bật tải thật v1/v2, chỉ khi muốn dùng mạng.**

Quay về cell setup chung, đổi **duy nhất** `RUN_KHANG = False` thành
`RUN_KHANG = True`; giữ cờ của người khác False. Chạy lại setup bằng Shift+Enter,
rồi chạy cell fixture Khang (cell 1 ở bảng trên) và cell tải mạng (cell 4).
Hai cell test offline không bắt buộc phải chạy lại để tải thật.

Chờ cell tải hết `[*]`. Nó xử lý lần lượt v1 và v2, in đường dẫn đích cùng dictionary
kết quả. `status='success'` nghĩa có nguồn TeX; `no_tex` là nguồn đã xác minh không có TeX;
`failed` là tải/xử lý lỗi, cần đọc `error`. BERT ở lần đã kiểm thử trả `success` cho cả hai.

Máy hiện tại **đã có nguồn và receipt từ lần thử trước**, nên lần đầu bạn bấm có thể
thấy `used_cache=True`, `attempts=0` ngay. Đây là cache hợp lệ, không phải tải mới.
Muốn thử tải mới, chỉ đổi `use_cache=True` thành `use_cache=False` ở lời gọi gán cho
`khang_live` trong cell mạng; giữ lời gọi gán cho `khang_cached` là `use_cache=True`.
Chạy lại cell mạng; sau khi kiểm xong trả lời gọi đầu về `use_cache=True`.
Tải mới có chờ trước request; không bấm chạy lặp khi cell còn `[*]`.

**Bước 9 — Mở nguồn thật và receipt.**

Trong cây file Jupyter, mở `data/manual/khang/live/1810-04805/sources/`:

```text
sources/
  v1/main.tex
  v1/sections/...
  v1/figures/...tex
  v2/main.tex
  v2/...tex
  .v1.receipt.json
  .v2.receipt.json
```

Mở `v1/main.tex`, `v2/main.tex` và các file `.tex` con, tìm `\input`, `\includegraphics`,
`figure`, `tabular` trong những file chứa chúng. Một lệnh không nhất thiết nằm ngay trong
`main.tex`. v1 giữ cấu trúc con; v2 có cấu trúc phẳng theo gói gốc. Các PDF ảnh đã bị bỏ.
Mở trang [BERT v1](https://arxiv.org/abs/1810.04805v1) và
[BERT v2](https://arxiv.org/abs/1810.04805v2) để xác nhận đúng ID/version.

Receipt có dấu chấm đầu tên nên có thể bị Jupyter ẩn. Cách chắc chắn trên máy này:
bấm **Win+E**, dán đường dẫn sau vào thanh địa chỉ Explorer rồi Enter:

```text
C:\Users\KhangTran\Downloads\HCMUS\Nhap_Mon_KHDL\arXivProject\data\manual\khang\live\1810-04805\sources
```

Mở `.v1.receipt.json` / `.v2.receipt.json` bằng VS Code hoặc Notepad. Kiểm `arxiv_id`,
`version`, `source_url` có hậu tố đúng version, `source_kind`, `result` và `files_sha256`.
Receipt ở ngoài thư mục v1/v2, không tính vào byte/file_count nguồn.

Kết quả đối chiếu lần đã thử: v1 **19 file**, 403593 byte trước lọc và 139057 byte sau lọc;
v2 **24 file**, 552453 byte trước và 173865 byte sau. Xem bảng bằng chứng phía dưới để phân biệt
byte gói nén với byte nguồn. Các con số này là dữ liệu thử BERT, không hardcode trong crawler.

**Bước 10 — Kiểm cache và xem bằng chứng.**

Giữ `RUN_KHANG=True` trong phiên làm việc, đặt lời gọi `khang_live` về `use_cache=True`,
chạy lại cell mạng. Khi file/receipt còn đúng, cả hai kết quả phải có
`used_cache=True`, `attempts=0`; nội dung nguồn không đổi. Cell còn gọi kiểm cache lần nữa
và assert hai điều này tự động.

Kiểm cache hỏng bằng **cell Fake HTTP**: test `test_bad_cache_refresh` cố ý sửa/xóa/thêm file,
làm hỏng receipt trong thư mục test riêng rồi kiểm phải tải lại. `OK` nghĩa ca này đạt.
Bản sao BERT thật cũng đã được thử sửa rồi tải lại; xem `cache-refresh-result.json` dưới đây.
Không cần sửa trực tiếp nguồn tốt để hoàn thành lượt kiểm tra cơ bản.

Trong `data/manual/khang/live/`, mở:

- `results.json`: hai DownloadResult của lần tải thật đã ghi bằng chứng.
- `audit.json`: hash đối chứng mọi file giữ lại, danh sách ảnh bỏ và kết quả cache.
- `cache-refresh-result.json`: lần thử cache BERT bị sửa và tải lại thành công.
- `control-v1.tar.gz`, `control-v2.tar.gz`: gói đối chứng tải riêng, có thể mở bằng phần mềm
  quản lý archive để xem danh sách/text gốc. Không dùng PDF bài báo thay nguồn TeX đối chứng.

Các file bằng chứng trên là **snapshot của lần kiểm trước**, không tự cập nhật mỗi lần bấm
cell tải. Kết quả lần bạn vừa chạy nằm ở output cell và `khang_live_results`; receipt được ghi
khi tải mới thành công. Dữ liệu thử nằm trong `data/`, đã được Git ignore.

**Bước 11 — Trả notebook về trạng thái bàn giao và tắt Jupyter.**

1. Trong setup, trả `RUN_KHANG=False`; các cờ RUN khác cũng False. Nếu đã đổi lời gọi tải,
   trả `use_cache=True` như mặc định.
2. Chọn **Edit → Clear All Outputs** (hoặc tìm `Clear All Outputs` trong Command Palette).
   Có thể dùng **Kernel → Restart Kernel and Clear Outputs…** nếu phiên bản Jupyter có mục đó.
   Kiểm output đã trống và số thực thi cell đã được xóa; bấm **Ctrl+S** lưu notebook.
3. Quay lại terminal Ubuntu đang chạy Jupyter, bấm **Ctrl+C**. Nếu hỏi shutdown thì nhập `y`, Enter.
4. Khi dấu nhắc terminal xuất hiện lại, chạy `conda deactivate`. Có thể đóng Ubuntu.

Lần sau chỉ cần lặp bước 1–3 để mở lại. Không cần tạo lại môi trường hay cài package lại.

---

### Khang: bằng chứng đã chạy và bàn giao

**Đã thử ngày 15/10/2026**, Ubuntu 24.04 WSL, Conda `arxiv-project`, Python 3.11.17,
requests 2.32.5; kiểm thử offline cũng chạy trên Windows Python 3.13.
**15 nhóm kiểm thử đạt; toàn bộ 13 code cell chạy đạt trong kernel Jupyter sạch**, với mạng tắt mặc định.
Mock đồng hồ chỉ áp dụng trong module tải, không thay sleep của luồng nền Jupyter.

| Hàm/case | Input + nguồn đối chiếu | Mong đợi | Thực tế | Kết luận + giới hạn |
|---|---|---|---|---|
| `prepare_source` | TeX UTF-8/CRLF, gzip đơn, tar và tar.gz gồm tex/bib/bbl/sty/cls/README/Latin-1/SVG + PNG/JPEG | Text giữ nguyên byte/cây thư mục; ảnh nhị phân bị bỏ; lệnh hình/bảng còn | Hash từng file text khớp, tổng byte trước/sau và số file khớp | Pass, gói kiểm thử tổng hợp |
| Không có TeX | PDF có xref, gzip PDF, tar chỉ PDF, DOCX | `no_tex`, output rỗng, byte trước chưa nén | Khớp | Pass local; chưa tải một bài PDF-only thật |
| Dữ liệu hỏng | Gzip cụt/CRC, tar cụt, PDF cụt, ZIP hỏng, HTML, byte ngẫu nhiên | `failed`, số chưa đo là `None`, giữ output cũ | Khớp | Pass offline |
| An toàn và dung lượng | Traversal/absolute/Windows path, symlink/hardlink/FIFO, tên trùng/case collision, giới hạn file/gói/giải nén/số member | Từ chối trước commit, không thoát đích | Khớp; nguồn tốt cũ còn nguyên | Pass offline, hạ giới hạn bằng mock để không tạo gói khổng lồ |
| Retry/pacing | HTTP giả 429 rồi 200, timeout rồi 200, 503 hai lần; Retry-After số/ngày | Chờ trước mọi request, tôn trọng Retry-After, attempts chính xác | Khớp; Retry-After 70 giây không bị cắt bởi cap 60 | Pass fake HTTP và fake sleep; không gây 429 thật |
| Cache | Receipt thiếu/hỏng/sai ID/version/schema/kind/stats/hash; file đổi/thiếu/thừa | Cache sai tải lại; cache tốt attempts=0/used_cache=True | Khớp; PDF no_tex cũng dùng receipt đã xác minh | Pass offline và cache BERT thật; sửa bản sao main.tex v1 đã khiến tải thật lại (attempts=1, used_cache=False), phục hồi hash gốc |
| Giữ output tốt | Timeout hết retry, 404/redirect/HTML, ghi receipt lỗi giả lập | `failed`; nguồn/receipt cũ giữ nguyên | Khớp; lỗi không chứa nội dung exception nhạy cảm | Pass; không thử mất điện/kill giữa rename |
| Mã arXiv cũ | `hep-th/9901001`, v2 qua HTTP giả | Giữ ID có slash, URL có v2 | Khớp | Chỉ kiểm cấu trúc URL; chưa tải thật mã cũ |
| BERT v1 | [Trang v1](https://arxiv.org/abs/1810.04805v1), [nguồn export v1](https://export.arxiv.org/src/1810.04805v1) | Tải đúng version, text khớp gói đối chứng | success; 19 file; bỏ 3 PDF ảnh; 19/19 SHA-256 khớp | Pass, tải thật và tải gói đối chứng riêng |
| BERT v2 | [Trang v2](https://arxiv.org/abs/1810.04805v2), [nguồn export v2](https://export.arxiv.org/src/1810.04805v2) | Thư mục riêng, không lẫn v1 | success; 24 file; bỏ 4 PDF ảnh; 24/24 SHA-256 khớp | Pass, tải thật và tải gói đối chứng riêng |

Hai lần tải thật đều `attempts=1`, `used_cache=False`. Thời gian gồm chờ trước request,
tải, xử lý và ghi: v1 **4.832 s**, v2 **5.263 s**. Đây là phép thử phát triển local,
**không phải benchmark Colab CPU** hoặc thống kê corpus chính thức.

| Version | archive_bytes (gói tải) | source_bytes_before (chưa nén) | source_bytes_after | file_count |
|---|---:|---:|---:|---:|
| v1 | 232356 | 403593 | 139057 | 19 |
| v2 | 316748 | 552453 | 173865 | 24 |

**Giới hạn đã công bố:** 256 MiB/gói tải, 512 MiB sau bung gzip (gồm framing tar), 64 MiB/file,
10.000 tar headers/members và 1 MiB/metadata header. Vượt giới hạn trả `failed`.
Nhận diện TeX/text dựa vào nội dung; giữ UTF-8 và text 8-bit không có control byte bất thường,
không chuyển encoding. SVG/EPS dạng text có thể được giữ; ảnh nhị phân/PDF bị bỏ.
UTF-16, ZIP chứa TeX, `.doc` OLE cũ và định dạng lạ chưa được hỗ trợ: trả `failed`, không tự gọi `no_tex`.
PDF chỉ được nhận diện bằng cấu trúc header/object/startxref/EOF, không kiểm đầy đủ mọi object hay render.
DOCX kiểm ZIP/CRC và root XML. Không chạy TeX, không xác minh nội dung khoa học của bài.
Redirect HTTP bị từ chối để tránh mất version hoặc gửi request ngoài nhịp.
Chỉ hỗ trợ ghi tuần tự vào một version_dir. Nguồn được stage và thay bằng rename, receipt ghi sau;
lỗi thông thường rollback. Kill/mất điện giữa các rename có thể để lại thư mục `.khang-*` chứa backup;
receipt thiếu khiến lần sau tải lại. Chưa kiểm thử crash recovery/concurrent writers, corpus được giao,
tích hợp các module, Colab CPU, Jupyter Book hoặc nộp bài.

**Methods and limitations — draft for the English report.**
We download explicit version identifiers from the arXiv export `/src/` endpoint using Requests,
with timeouts, sequential pacing, bounded retries and Retry-After handling. The transport body is
stored without automatic HTTP content decoding so downloaded bytes remain distinct from uncompressed
source bytes. We identify gzip, tar, standalone TeX, PDF and DOCX by their contents. Tar headers and
paths are validated before writing files; links, special files, collisions and oversized inputs are
rejected. Text files are copied byte for byte with their relative paths, preserving figure/table
commands while discarding binary assets. Both source-size measurements exclude archive framing,
directories and receipts. A complete source without identifiable TeX produces an empty version record;
unknown, damaged or inaccessible input is a failure. Version-specific receipts store the URL, source
kind, measurements and SHA-256 digest of every retained file. Cache reuse requires matching identity,
schema, file inventory and hashes. Failed refreshes preserve prior outputs. Validation includes local
adversarial packages, mocked HTTP failures, and independent source-archive comparisons for BERT v1/v2.
These development checks are not a corpus-wide evaluation or a Colab benchmark. Detection is
conservative and does not compile TeX or fully validate PDF objects; unsupported formats and over-limit
submissions fail explicitly. Concurrent writers and crash recovery require further work.

References checked 2026-10-15: [arXiv media types](https://info.arxiv.org/help/mimetypes.html),
[arXiv API terms and pacing](https://info.arxiv.org/help/api/tou.html), plus the versioned pages/sources above.


## 3. Nguyên — Metadata tham khảo, không cần nguồn Khang
DateLookup nhận list ID và trả DateLookupResult (dates + errors).
Khi Khánh chưa xong, dùng DATE_RESULT để biết schema output hàm tra ngày.
Khi ghép thật, functools.partial gắn policy vào fetch_submission_dates.
Không dùng publication_date thay submission_date.


**Quy trình Nguyên:** S2 đủ trang → lọc arXiv → date_lookup → references.json. Khi Khánh chưa xong, fake DateLookup phải trả đúng dates/errors và chỉ dùng dữ liệu mẫu được khai báo. Thay bằng partial hàm thật trước chạy corpus. Kiểm tra 429/not_found/thiếu ngày và không ghi đè file tốt bằng lỗi.


In [ ]:
nguyen_id = fixtures.PAPER_BERT['arxiv_id']
nguyen_dates_sample = deepcopy(fixtures.DATE_RESULT)
nguyen_references_sample = deepcopy(fixtures.REFERENCE_RESULT)
assert nguyen_references_sample['arxiv_id'] == nguyen_id
assert nguyen_references_sample['kept_references'] == len(nguyen_references_sample['references'])
assert nguyen_dates_sample['dates']['1706.03762'] == nguyen_references_sample['references']['1706-03762']['submission_date']
nguyen_references_sample


In [ ]:
# Helper TEST ONLY: không dùng fake này để chạy corpus nộp.
def nguyen_fake_dates(ids):
    known = fixtures.DATE_RESULT['dates']
    dates = {i: known.get(i) for i in dict.fromkeys(ids)}
    return {'dates': dates,
            'errors': {i: 'Fixture không có ngày; chưa tra API thật.' for i, value in dates.items() if value is None}}

if RUN_NGUYEN:
    nguyen_date_lookup = partial(fetch_submission_dates, policy=POLICY) if USE_REAL_DATES else nguyen_fake_dates
    nguyen_paper_dir = TEST_ROOT / 'nguyen' / '1810-04805'
    nguyen_had_previous_file = (nguyen_paper_dir / 'references.json').exists()
    nguyen_live = collect_references(nguyen_id, nguyen_paper_dir, nguyen_date_lookup,
                                    policy=POLICY, use_cache=False)  # buộc refresh khi đổi fake -> thật
    assert nguyen_live['arxiv_id'] == nguyen_id
    assert nguyen_live['kept_references'] == len(nguyen_live['references'])
    assert nguyen_live['missing_submission_dates'] <= nguyen_live['incomplete_references'] <= nguyen_live['kept_references']
    if nguyen_live['status'] == 'success' or (nguyen_live['status'] == 'partial' and not nguyen_had_previous_file):
        import json
        assert json.loads((nguyen_paper_dir / 'references.json').read_text(encoding='utf-8')) == nguyen_live['references']
    print('Ngày thật:', USE_REAL_DATES, nguyen_live)
else:
    print('Nguyên: đang tắt gọi hàm thật. Đọc mục 14 phân công trước khi bật.')


### Nguyên: kiểm tra tay và ghi bằng chứng

Đọc mục 14 của `docs/milestone1/PHAN_CONG_MILESTONE_1_4_NGUOI.md`.
Restart Kernel → chạy setup → các ô phần này là đủ, không cần fixture của phần khác.
Các assert trên chỉ kiểm một số bất biến; còn phải thêm ca lỗi/schema và mở đối chiếu nguồn/file.

| Ngày/môi trường | Hàm/case | Input/nguồn đối chiếu | Mong đợi | Thực tế/file | Kết luận/giới hạn |
|---|---|---|---|---|---|
| Chưa chạy thật | Điền sau triển khai | — | — | — | Chưa kiểm chứng |

Giữ mặc định cờ RUN=False trước commit, clear output. Không ghi pass cho ca chưa chạy.


## 4. Danh — Ghép pipeline, thống kê và benchmark
Dùng MANIFEST để làm summarize_corpus trước khi có module thật.
run_pipeline cho phép truyền dependency cùng chữ ký, chưa cung cấp mock xử lý sẵn.
Chỉ pipeline ghi manifest. Bộ đo lấy mẫu phải chạy trong lúc crawler chạy, kể cả tiến trình con.


**Quy trình Danh:** summarize_corpus/benchmark với fixture → manifest/file → pipeline/resume → bộ đo → tích hợp thật → Colab CPU → Book. Kiểm tra các kết quả kỳ vọng bên dưới; fixture xanh chỉ kiểm tra hợp đồng, không là benchmark thật.


In [ ]:
danh_manifest_sample = deepcopy(fixtures.MANIFEST)
danh_id_sample = fixtures.PAPER_BERT['arxiv_id']
assert danh_manifest_sample['discovery']['papers'][0] == fixtures.PAPER_BERT
assert set(danh_manifest_sample['papers'][danh_id_sample]['downloads']) == {'1', '2'}
assert all(value['status'] == 'success' for value in danh_manifest_sample['papers'][danh_id_sample]['downloads'].values())
# Tính tay ở mục 15; fixture không phải số liệu corpus thật.
fixtures.CORPUS_SUMMARY


In [ ]:
if RUN_DANH_METRICS:
    danh_summary = summarize_corpus(deepcopy(fixtures.MANIFEST))
    assert danh_summary == fixtures.CORPUS_SUMMARY
    danh_benchmark = summarize_benchmark(deepcopy(fixtures.RESOURCE_SAMPLES), wall_seconds=20.0,
        final_disk_bytes=1000, environment=fixtures.BENCHMARK['environment'], cpu_only=True,
        cache_mode='cold', sampling_interval_seconds=10.0)
    assert danh_benchmark == fixtures.BENCHMARK
    print(danh_summary, danh_benchmark)
else:
    print('Danh: đang tắt kiểm hàm metrics thật. Thêm ca lỗi/resume theo mục 15.')

if RUN_INTEGRATION:
    # Chỉ bật khi cả bốn module thật xong. Chưa dùng phạm vi chính thức.
    danh_run = run_pipeline(deepcopy(fixtures.SCOPE), TEST_ROOT / 'integration',
                            policy=POLICY, resume=True)
    print(danh_run)
    # Mở file và đối chiếu theo mục 15; không coi in ra status là đủ nghiệm thu.


### Danh: kiểm tra tay và ghi bằng chứng

Đọc mục 15 của `docs/milestone1/PHAN_CONG_MILESTONE_1_4_NGUOI.md`.
Restart Kernel → chạy setup → các ô phần này là đủ, không cần fixture của phần khác.
Các assert trên chỉ kiểm một số bất biến; còn phải thêm ca lỗi/schema và mở đối chiếu nguồn/file.

| Ngày/môi trường | Hàm/case | Input/nguồn đối chiếu | Mong đợi | Thực tế/file | Kết luận/giới hạn |
|---|---|---|---|---|---|
| Chưa chạy thật | Điền sau triển khai | — | — | — | Chưa kiểm chứng |

Giữ mặc định cờ RUN=False trước commit, clear output. Không ghi pass cho ca chưa chạy.


## 5. Mẫu tình huống lỗi
Fixture giúp thống nhất ý nghĩa status, không chứng minh module đã xử lý lỗi đúng.
Khi viết thật cần kiểm tra PDF-only, gói hỏng/traversal, mã cũ, thiếu trang S2,
429/timeout, date thiếu, no_tex, input sai, resume và ghi file nguyên tử.


In [ ]:
# Output cuối kỳ vọng cho manifest giả lập, KHÔNG phải run thật.
expected_run = deepcopy(fixtures.RUN_RESULT)
assert expected_run['summary']['versions_success'] == 2
expected_run


In [ ]:
assert fixtures.DOWNLOAD_NO_TEX['status'] == 'no_tex'
assert fixtures.DOWNLOAD_FAILED['status'] == 'failed'
assert fixtures.DOWNLOAD_FAILED['source_bytes_after'] is None
assert fixtures.REFERENCE_FAILED['total_references'] is None
assert not fixtures.REFERENCE_FAILED['pagination_complete']
assert fixtures.DATE_RESULT_MISSING['dates']['1706.03762'] is None
print('Mẫu input/output nhất quán. Chưa chạy hay kiểm thử crawler thật.')


## 6. Hoàn thành đồ án sau triển khai

- Chạy đúng phạm vi thầy giao, ghi đủ mọi phiên bản kể cả không có TeX/lỗi.
- metadata.json và references.json đầy đủ, nguồn văn bản không bị đổi; giữ lệnh hình/bảng.
- Thống kê chất lượng dữ liệu và benchmark end-to-end Colab CPU, có cache mode và mẫu số.
- Báo cáo tiếng Anh trên Jupyter Book repo riêng; bảng/biểu đồ sinh lúc build từ dữ liệu lưu.
- README cài/chạy, requirements đã kiểm chứng, code công khai; không commit corpus hay key.
- Nộp link code + báo cáo qua Moodle đúng hạn. Giữ corpus cho Milestone 2–3.

Đọc thêm docs/milestone1/HUONG_DAN_THANH_VIEN_MILESTONE_1.md. Bộ khung này không thay việc triển khai,
benchmark, báo cáo và nộp bài. Không dùng fixture làm số liệu thật.
